In [1]:
from pathlib import Path
import sys
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")
print(PROJECT_ROOT)

C:\Users\amanm\Desktop\learning\developer-chat-agent


In [2]:
from src.config import parent_store_collection
from src.generation.generator import generate_answer
from src.retrieval.retriever import search_vector_db
from src.caching.semantic_cache import get_semantic_cache, set_semantic_cache
from langchain_groq import ChatGroq
from src.config import OPENAI_MODEL_GROQ, TEMPERATURE, MAX_TOKENS, GROQ_API_KEY



c:\Users\amanm\Desktop\learning\developer-chat-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index 'agenticrag' already exists


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7543.45it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
llm_evaluator_for_retrieval = ChatGroq(
    model=OPENAI_MODEL_GROQ,
    api_key=GROQ_API_KEY,
    temperature=0,
    max_tokens=MAX_TOKENS,
)


In [4]:
def llm_router(dict):
    prompt = f"""
    You are a strict document evaluator for a RAG system.
    You're task is to rate each query and doc pair and tell if it is "good" or "bad".
    Answer only "good" or "bad"


Query : {dict['query']}
Doc : {dict['doc']}

Answer:
    """
    return llm_evaluator_for_retrieval.invoke(prompt).content.strip()


In [5]:
def get_documents_for_evaluation(query, namespace: str = "default_namespace"):
    retrieved_chunks = search_vector_db(
        namespace=namespace, 
        query=query, 
        top_k=3
    )
    
    # Get unique parent IDs
    parent_ids = {chunk.get("parent_id") for chunk in retrieved_chunks if chunk.get("parent_id")}
    
    documents = []
    if parent_ids:
        parent_docs = list(parent_store_collection.find({
            "parent_id": {"$in": list(parent_ids)},
            "namespace": namespace
        }))
        for doc in parent_docs:
            documents.append(doc["text"])
    else:
        # fallback to child chunks if parent_child is off
        for chunk in retrieved_chunks:
            documents.append(chunk["chunk_text"])
            
    return documents


In [6]:
query = "Tell me about Cross-Modal Knowledge Graph"
documents = get_documents_for_evaluation(query)
print(f"Retrieved {len(documents)} documents.")


Retrieved 3 documents.


In [7]:
import re

def clean_and_split(text: str):
    # 1. Fix broken words
    text = re.sub(r'-\s*\n\s*', '', text)
    # 2. Replace newlines with space
    text = text.replace('\n', ' ')
    # 3. Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # 4. Split into knowledge strips (sentences)
    sentences = re.split(r'(?<=[.!?])\s+', text)
    # 5. Remove short noisy strips
    return [s.strip() for s in sentences if len(s.strip()) > 20]


In [8]:
# Mock Web Search fallback simulating an external knowledge step
def fallback_web_search(query):
    print("\n=> Triggering web search fallback for query:", query)
    return "[Web Search Results]: Cross-Modal Knowledge Graphs integrate text, images, and other modalities."

final_context_strips = []
needs_web_search = True

for i, doc in enumerate(documents):
    # 1. Document Evaluation
    eval_result = llm_router({"query": query, "doc": doc})
    print(f"Document {i+1} Evaluation: {eval_result}")
    
    if eval_result.lower() == "good":
        # Document is Correct -> Knowledge Refinement
        needs_web_search = False
        strips = clean_and_split(doc)
        
        # 2. Strip Evaluation
        for strip in strips:
            strip_eval = llm_router({"query": query, "doc": strip})
            if strip_eval.lower() == "good":
                final_context_strips.append(strip)
    else:
        # Document is Incorrect
        pass

# 3. Fallback to Web Search if no valid internal documents
if needs_web_search or len(final_context_strips) == 0:
    external_knowledge = fallback_web_search(query)
    final_context_strips.append(external_knowledge)

final_context = "\n".join(final_context_strips)
print("\n--- Final Refined Context ---")
print(final_context)


Document 1 Evaluation: good
Document 2 Evaluation: good
Document 3 Evaluation: good

--- Final Refined Context ---
• Cross-Modal Knowledge Graph: Non-textual content like images, tables, and equations contains rich semantic information that traditional text-only approaches often overlook.
The resulting text-based knowledge graph captures explicit knowledge and semantic connections present in textual portions of documents, complementing the multimodal graph’s cross-modal grounding capabilities.
2.2.2 GRAPH FUSION AND INDEX CREATION The separate cross-modal and text-based knowledge graphs capture complementary aspects of document semantics.


In [9]:
from src.generation.generator import llm

messages = [
    ("system", "You are an assistant. Answer the query based on the context."),
    ("human", f"Context:\n{final_context}\n\n---\nQuestion: {query}")
]
response = llm.invoke(messages)
print("\n--- Final Generated Answer ---")
print(response.content)



--- Final Generated Answer ---
**Cross‑Modal Knowledge Graph (CMKG)**  

A Cross‑Modal Knowledge Graph is a structured representation that links **semantic information from multiple modalities**—typically text, images, tables, and mathematical equations—into a single, query‑able graph. Unlike traditional knowledge graphs that are built only from textual sources, a CMKG explicitly models the meaning that resides in non‑textual content and ties it to the surrounding narrative.

---

### Why a CMKG is Needed  

| Modality | What it contributes | What is missed by text‑only KG |
|----------|--------------------|--------------------------------|
| **Images** | Objects, scenes, visual relationships, visual attributes (color, shape, layout) | Visual cues, visual metaphors, diagrams |
| **Tables** | Structured numeric or categorical data, relational patterns, summary statistics | Precise quantitative relationships, row/column semantics |
| **Equations / Formulas** | Mathematical relationships